## Simple Web Query
Here we can use the Python SDK to develop the simple web query agent, then save the agent to a config.yaml and run it from there.

In [ ]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [ ]:
from nat.agent.sdk import NatReActAgent
from nat.embedder.sdk import NIMEmbedder
from nat.llm.sdk import NimLLM
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow
from nat_simple_web_query.sdk import WebQueryTool

llm = NimLLM(
    model_name="nvdev/meta/llama-3.1-70b-instruct",
    temperature=0.0,
    name="nim_llm",
)

embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",
    name="nv-embedqa-e5-v5",
)

current_time_tool = CurrentTimeTool(
    name="current_datetime",
)

web_query_tool = WebQueryTool(
    webpage_url="https://docs.smith.langchain.com",
    description="Search for information about LangSmith. For any questions about LangSmith, you must use this tool!",
    embedder=embedder,
    chunk_size=512,
    name="webpage_query",
)

agent = NatReActAgent(
    tools=[web_query_tool, current_time_tool],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=3,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)

In [ ]:
await nat_workflow.prompt('What is LangSmith?')

In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())